# 02 — Zero-shot BART-MNLI trên GoEmotions

Notebook giải thích phần B và đọc trạng thái/kết quả thật. Chạy tất cả các ô **không tự tải model, huấn luyện hoặc chạy full**. Lệnh được in để bạn chủ động chạy ở terminal. Mở bằng kernel của môi trường đã cài extra Transformer; xem `docs/GETTING_STARTED.md`.

Thiếu run → báo chưa có. Run đang chạy → báo batch đã lưu. Run complete → kiểm hash/mapping rồi hiện metric. JSON lỗi, hash sai hoặc artifact bị mất sẽ báo lỗi, không chuyển thành “chưa xong”.

## 1. Bài toán

Văn bản tiếng Anh → 28 cặp văn bản/giả thuyết → điểm NLI cho mỗi nhãn → ngưỡng → tập nhãn. Ví dụ giả thuyết: `This text expresses joy.`. Điểm này chưa phải xác suất cảm xúc được hiệu chuẩn. `multi_label=True` cho phép nhiều nhãn cùng cao.

B dùng `facebook/bart-large-mnli` đã học MNLI; nhóm không cập nhật trọng số trên GoEmotions. A học TF-IDF/LR từ train; C fine-tune 3 kiến trúc.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src/models/zero_shot.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "src/models/zero_shot.py").is_file():
    raise RuntimeError("Mở notebook từ repo hoặc thư mục notebooks")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.datasets.goemotions import EXPECTED_ROWS, EXPECTED_SHA256, REVISION, sha256
from src.models.zero_shot import CHECKPOINT, HYPOTHESIS_TEMPLATE
from src.evaluation.protocols import check_full_run
labels = json.loads((ROOT / "data/labels.json").read_text(encoding="utf-8"))
if len(labels) != 28 or len(set(labels)) != 28:
    raise ValueError("Cần đúng 28 nhãn GoEmotions")
print("Repo:", ROOT)
print("Kernel Python:", sys.executable)
print("Số nhãn:", len(labels))

## 2. Vì sao phải đổi thứ tự nhãn?

Pipeline xếp nhãn theo điểm; mỗi câu có thể xếp khác nhau. Ví dụ nhỏ dưới đây chỉ giải thích việc ghép cột, không phải đầu ra BART đã đo.

In [ ]:
from src.models.zero_shot import map_pipeline_scores

example = [{"labels": ["anger", "joy"], "scores": [0.9, 0.2]},
           {"labels": ["joy", "anger"], "scores": [0.8, 0.1]}]
print(map_pipeline_scores(example, ["joy", "anger"]))
# Cột 0 luôn là joy, cột 1 luôn là anger.

## 3. Kiểm tra nhỏ trước khi chạy đủ

Máy CUDA 8 GB dùng `--dtype float16 --batch-size 16`; CPU dùng `--dtype float32`. Smoke dùng 8 câu, lưu folder riêng và không phải kết quả đầy đủ.

Notebook chỉ **in lệnh**, không tự chạy GPU. Nếu folder smoke đã có cấu hình khác, giữ nó làm log lịch sử; không trộn hoặc xóa để che lỗi resume.

In [ ]:
# Chọn lệnh đúng thiết bị; không chạy cả hai vào cùng folder smoke.
print("CUDA:")
print("python -m scripts.zero_shot.run_zero_shot --smoke --device cuda --dtype float16 --batch-size 16")
print("CPU:")
print("python -m scripts.zero_shot.run_zero_shot --smoke --device cpu --dtype float32 --batch-size 16")

## 4. Full validation và resume

Máy CUDA đang dùng: `python -m scripts.zero_shot.run_zero_shot --device cuda --dtype float16 --batch-size 16`.
Máy CPU chạy một thí nghiệm riêng với `--device cpu --dtype float32`.

Full cần **5.426 câu × 28 nhãn**. Chạy lại cùng lệnh để tiếp batch đã ghi. Giữ nguyên model SHA, dtype, batch-size, thiết bị và môi trường; không chuyển run CUDA float16 sang CPU float32 để resume. `--limit` luôn là smoke.

Ô dưới đọc metadata và batch manifest, **không gọi pipeline**. Batch đã lưu được kiểm SHA/shape/nhãn. Nếu metadata hoàn tất, kiểm full scores rồi hiển thị chỉ số; không lấy số của batch dở làm F1 full.

In [ ]:
def read_b_status(output, label_names):
    """Chỉ đọc local artifacts; lỗi dữ liệu/hashes được giữ nguyên để sửa."""
    output = Path(output)
    manifest_path = output / "validation_checkpoints/checkpoint_manifest.json"
    metadata_path = output / "run_metadata.json"
    status = {"status": "chưa có run full", "saved_samples": 0,
              "total_samples": EXPECTED_ROWS["validation"], "dtype": None, "device": None}
    manifest = None
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        config = manifest["config"]
        if (manifest.get("schema_version") != 1 or config.get("method") != "zero_shot"
                or config.get("smoke") is not False or config.get("split") != "validation"
                or config.get("data_revision") != REVISION
                or config.get("data_sha256", {}).get("validation") != EXPECTED_SHA256["validation"]
                or config.get("label_names") != label_names
                or config.get("sample_count") != EXPECTED_ROWS["validation"]):
            raise ValueError("Manifest B không khớp thí nghiệm full validation")
        if manifest.get("status") not in ("running", "complete"):
            raise ValueError("Trạng thái manifest B không hợp lệ")
        completed, seen_ids = 0, set()
        for chunk in manifest["chunks"]:
            filename = f"chunk_{chunk['start']:06d}_{chunk['stop']:06d}.npz"
            if (chunk["file"] != filename or chunk["start"] != completed
                    or not completed < chunk["stop"] <= config["sample_count"]):
                raise ValueError("Thứ tự/tên chunk B sai")
            path = manifest_path.parent / filename
            if sha256(path) != chunk["sha256"]:
                raise ValueError(f"Chunk B sai hash: {filename}")
            with np.load(path, allow_pickle=False) as saved:
                points = saved["scores"]
                if (points.shape != (chunk["stop"] - completed, 28)
                        or saved["label_names"].astype(str).tolist() != label_names
                        or saved["ids"].shape != (len(points),)
                        or len(set(saved["ids"].astype(str))) != len(points)
                        or not np.isfinite(points).all()
                        or np.any((points < 0) | (points > 1))):
                    raise ValueError(f"Chunk B sai shape/ID/nhãn/điểm: {filename}")
                chunk_ids = set(saved["ids"].astype(str))
                if seen_ids & chunk_ids:
                    raise ValueError("ID bị lặp giữa các chunk B")
                seen_ids.update(chunk_ids)
            completed = chunk["stop"]
        if manifest["status"] == "complete" and (
                completed != config["sample_count"] or manifest.get("completed_count") != completed):
            raise ValueError("Manifest B complete nhưng số batch/mẫu chưa đủ")
        status.update(status="đang chạy hoặc bị ngắt; chưa có kết quả full",
                      saved_samples=completed, dtype=config.get("dtype"), device=config.get("device"))
    if not metadata_path.exists():
        return status, None
    metadata = check_full_run(output, label_names)  # Từ chối smoke/hash sai/mapping sai.
    if (metadata.get("sample_count") != EXPECTED_ROWS["validation"]
            or metadata.get("split") != "validation" or manifest is None
            or manifest.get("status") != "complete"
            or status["saved_samples"] != EXPECTED_ROWS["validation"]
            or metadata.get("checkpoint_manifest_sha256") != sha256(manifest_path)):
        raise ValueError("B metadata complete nhưng batch/số mẫu chưa khớp")
    with np.load(output / "validation_scores.npz", allow_pickle=False) as saved:
        points, ids = saved["scores"], saved["ids"].astype(str)
        if (points.shape != (EXPECTED_ROWS["validation"], 28)
                or ids.shape != (len(points),) or len(set(ids)) != len(ids)
                or saved["label_names"].astype(str).tolist() != label_names
                or not np.isfinite(points).all() or np.any((points < 0) | (points > 1))):
            raise ValueError("B full scores không hợp lệ")
    for field in ("checkpoint", "model_revision", "hypothesis_template", "device", "dtype"):
        if metadata.get(field) != manifest["config"].get(field):
            raise ValueError(f"Metadata và manifest B khác {field}")
    metrics = metadata["metrics"]
    if metrics["n_samples"] != len(points) or metrics["n_labels"] != 28:
        raise ValueError("B metrics khác số mẫu/nhãn trong scores")
    status.update(status="complete: full validation đã kiểm hash", dtype=metadata.get("dtype"),
                  device=metadata.get("device"), model_revision=metadata["model_revision"])
    return status, metrics

b_status, b_metrics = read_b_status(ROOT / "data/processed/zero_shot/full", labels)
display(pd.DataFrame([b_status]))
if b_metrics is not None:
    display(pd.DataFrame([{name: b_metrics[name] for name in
                          ("macro_f1", "micro_f1", "macro_precision", "macro_recall",
                           "micro_precision", "micro_recall", "hamming_loss")}]))
else:
    print("Chưa có metrics full. Số batch đã lưu không phải điểm chất lượng mô hình.")

## 5. Khóa ngưỡng trước test

Ngưỡng @0,5 là mốc B gốc. Ngưỡng global/per-label chọn bằng nhãn validation phải ghi **trọng số zero-shot, có hiệu chỉnh ngưỡng bằng validation**.

Chỉ sau full validation, tạo protocol một lần:

```powershell
python -m scripts.pipeline.freeze_experiment --run-dir data/processed/zero_shot/full
python -m scripts.zero_shot.run_zero_shot --split test --device cuda --dtype float16 --batch-size 16 --protocol data/processed/zero_shot/full/final_protocol.json
```

Test giữ dtype/device/batch-size của run validation. Máy CPU dùng float32 trong một run CPU riêng. Có protocol rồi thì giữ nguyên; không xóa/khóa lại để chọn bằng test. Notebook này không chạy các lệnh test.

## 6. Bàn giao và tự kiểm

- NPZ gồm ID, scores N×28 và label_names theo thứ tự chuẩn.
- Metadata chứa SHA checkpoint, data revision/hash, môi trường, thời gian và chỉ số shared.
- So sánh A/B/C phải cùng split, nhãn và metric; phân biệt validation/test.
- `python -m unittest tests.test_zero_shot -v` dùng dữ liệu giả kiểm mapping, resume, hash và protocol; không chứng minh chất lượng mô hình.

**Nguồn:** [model card](https://huggingface.co/facebook/bart-large-mnli), [HF pipeline](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines#transformers.ZeroShotClassificationPipeline), [GoEmotions ACL 2020](https://aclanthology.org/2020.acl-main.372/).